In [1]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/initialDB_sequences.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5\' UTR', 'start_codon', 'CDS', 'stop_codon', '3\' UTR', 'appris_rank', '5\' UTR_len', 'CDS_len', '3\' UTR_len'])

df = initialDB.copy()
df.head()

,gene_id,transcript_id,5' UTR,start_codon,CDS,stop_codon,3' UTR,appris_rank,5' UTR_len,CDS_len,3' UTR_len
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATG,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,TAG,NaN,1,60,981,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATG,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,TAA,1,0,939,3
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATG,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,TAA,1,0,939,3
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATG,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,TGA,NaN,10,509,2535,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATG,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGA,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,3,16,2250,494


In [2]:
def gc_content(seq):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    if not seq:
        return np.nan
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq)

df = initialDB.copy()
df['5_UTR_GC'] = df['5\' UTR'].fillna('').map(gc_content)
df['CDS_GC'] = df['CDS'].fillna('').map(gc_content)
df['3_UTR_GC'] = df['3\' UTR'].fillna('').map(gc_content)
df['global_GC'] = df[['5\' UTR', 'CDS', '3\' UTR']].fillna('').agg(''.join, axis=1).map(gc_content)

In [3]:
motifs = {
"ATTTA": r"A[TU]{3}A",
"WTTTW": r"[ACGTU][TU]{3}[ACGTU]",
"WWTTTWW": r"[ACGTU]{2}[TU]{3}[ACGTU]{2}",
"WWWTTTWWW": r"[ACGTU]{3}[TU]{3}[ACGTU]{3}",
"WWWWTTTWWWW": r"[ACGTU]{4}[TU]{3}[ACGTU]{4}",
"WWWWWTTTWWWWW": r"[ACGTU]{5}[TU]{3}[ACGTU]{5}",
"TTTGTTT": r"[TU]{3}G[TU]{3}",
"GTTTG": r"G[TU]{3}G",
"AWTAAA": r"A[ACGTU][TU]AAA"
}

def scan_motifs(df, sequence_col, count_prefix):
    """
    Scans a specific column for motifs and adds a total count column.
    Returns a detailed breakdown DataFrame for those specific motifs.
    """
    total_col_name = f"{count_prefix}_AREs"
    df[total_col_name] = 0
    
    motif_counts_dict = {
        'gene_id': df['gene_id'],
        'transcript_id': df['transcript_id'],
        f'{count_prefix}_sequence': df[sequence_col]
    }
    
    for name, pattern in motifs.items():
        counts = df[sequence_col].str.findall(pattern, flags=re.IGNORECASE).str.len().fillna(0).astype(int)
        
        df[total_col_name] += counts
        
        motif_counts_dict[f"{count_prefix}_{name}_count"] = counts
    
    detailed_df = pd.DataFrame(motif_counts_dict)
    return df, detailed_df

df, AREs_5_df = scan_motifs(df, '5\' UTR', '5')

df, AREs_3_df = scan_motifs(df, '3\' UTR', '3')

In [4]:
def scan_uorf(df, utr5_col):
    """
    Scans 5' UTR for:
    1. uAUGs: Any AUG in the 5' UTR
    2. uORF: AUG in frame with the canonical start codon 
    """

    def classify_utr_motifs(seq):
        if pd.isna(seq) or len(seq) < 3:
            return 0, 0
        
        seq = seq.upper()
        # uAUG scanning
        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]
        uaug_count = len(atg_indices)
        
        # uORF scanning
        uorf_count = 0
        seq_len = len(seq)
        for start_idx in atg_indices:
            distance_to_end = seq_len - start_idx
            if distance_to_end % 3 == 0:
                uorf_count += 1
                
        return uaug_count, uorf_count

    results = df[utr5_col].apply(classify_utr_motifs)
    df[['uAUG_count', 'uORF_in_frame']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

def scan_dorf(df, utr3_col):
    """
    Scans 3' UTR for dORFs: 
    Downstream ORFs starting with AUG in the 3' UTR.
    Returns count of dORFs that find a stop codon vs. those that don't (overlap/truncated).
    """

    def classify_dorf_motifs(seq):
        if pd.isna(seq) or len(seq) < 3:
            return 0, 0
        
        seq = seq.upper()
        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]
        
        non_overlap = 0 # Finding a stop codon within the 3' UTR
        overlap = 0     # Running off the end of the 3' UTR without a stop
        
        stops = {'TAA', 'TAG', 'TGA'}

        for start_idx in atg_indices:
            remaining_seq = seq[start_idx:]
            found_stop = False
            
            # Check every 3 bases from the current ATG
            for i in range(0, len(remaining_seq) - 2, 3):
                codon = remaining_seq[i:i+3]
                if codon in stops:
                    found_stop = True
                    break
            
            if found_stop:
                non_overlap += 1
            else:
                overlap += 1
                
        return non_overlap, overlap

    results = df[utr3_col].apply(classify_dorf_motifs)
    df[['dORF_with_stop', 'dORF_truncated']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

df = scan_uorf(df, '5\' UTR')
df = scan_dorf(df, '3\' UTR')

In [5]:
IRES_path = Path('../data/utr_ire_scan_results.csv')
IRES_db = pd.read_csv(IRES_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len', 'IRES count', 'IRES IDs'
])


cols_to_keep = [col for col in IRES_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, IRES_db[cols_to_keep], on='transcript_id', how='left')

In [6]:
mfe_path = Path('../data/mfe_results.csv')
mfe_db = pd.read_csv(mfe_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_len', 'CDS_len', '3_len', '5_UTR_MFE', 'CDS_MFE', '3_UTR_MFE'
])
### Note: MFE values of NaN were not able to be folded by RNAfold either because the sequence is length 0 or it's too long (over 100k nt) for RNAfold to handle.

cols_to_keep = [col for col in mfe_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, mfe_db[cols_to_keep], on='transcript_id', how='left')

In [7]:
# Protein Sequence 
codons = {
"TTT" : "F", "CTT" : "L", "ATT" : "I", "GTT" : "V", "TTC" : "F",
"CTC" : "L", "ATC" : "I", "GTC" : "V", "TTA" : "L", "CTA" : "L",
"ATA" : "I", "GTA" : "V", "TTG" : "L", "CTG" : "L", "ATG" : "M",
"GTG" : "V", "TCT" : "S", "CCT" : "P", "ACT" : "T", "GCT" : "A",
"TCC" : "S", "CCC" : "P", "ACC" : "T", "GCC" : "A", "TCA" : "S",
"CCA" : "P", "ACA" : "T", "GCA" : "A", "TCG" : "S", "CCG" : "P",
"ACG" : "T", "GCG" : "A", "TAT" : "Y", "CAT" : "H", "AAT" : "N",
"GAT" : "D", "TAC" : "Y", "CAC" : "H", "AAC" : "N", "GAC" : "D",
"TAA" : "Stop", "CAA" : "Q", "AAA" : "K", "GAA" : "E", "TAG" : "Stop", 
"CAG" : "Q", "AAG" : "K", "GAG" : "E", "TGT" : "C", "CGT" : "R",
"AGT" : "S", "GGT" : "G", "TGC" : "C", "CGC" : "R", "AGC" : "S",
"GGC" : "G", "TGA" : "Stop", "CGA" : "R", "AGA" : "R", "GGA" : "G",
"TGG" : "W", "CGG" : "R", "AGG" : "R", "GGG" : "G"
}

def translate_cds_to_protein(cds_seq):
    if pd.isna(cds_seq) or len(cds_seq) < 3:
        return np.nan
    
    cds_seq = cds_seq.upper()
    protein_seq = ''
    
    for i in range(0, len(cds_seq) - 2, 3):
        codon = cds_seq[i:i+3]
        amino_acid = codons.get(codon, 'X')  # 'X' for unknown codons
        if amino_acid == "Stop":
            break
        protein_seq += amino_acid
    
    return protein_seq

df['protein_sequence'] = df['CDS'].apply(translate_cds_to_protein)

In [8]:
# Load gencode protein translations keyed by transcript ID
from pathlib import Path

gencode_fasta_path = Path('..') / 'data' / 'gencode.v49.pc_translations.fa'

def load_protein_fasta_by_transcript(fasta_path):
    """Load FASTA and map transcript ID to amino acid sequence."""
    protein_dict = {}
    with open(fasta_path) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id is not None:
                    protein_dict[current_id] = ''.join(current_seq)
                header_parts = line[1:].split('|')
                current_id = header_parts[1] if len(header_parts) > 1 else header_parts[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            protein_dict[current_id] = ''.join(current_seq)
    return protein_dict

# load by transcript ID
mappings_by_transcript = load_protein_fasta_by_transcript(gencode_fasta_path)
print(f"Loaded {len(mappings_by_transcript)} transcript AA sequences from gencode")

Loaded 245535 transcript AA sequences from gencode


In [9]:
# Compare protein_sequence directly against the gencode sequence for the same transcript ID

def matches_gencode_by_transcript(row, gencode_by_transcript):
    """Return True if the translated protein sequence equals the gencode translation for the transcript."""
    aa_seq = row['protein_sequence']
    tid = row['transcript_id']
    if pd.isna(aa_seq) or aa_seq == '':
        return False
    return aa_seq == gencode_by_transcript.get(tid, '')

# Apply direct transcript-ID based matching
df['matches_gencode_aa'] = df.apply(lambda r: matches_gencode_by_transcript(r, mappings_by_transcript), axis=1)

print(f"\nMatches found: {df['matches_gencode_aa'].sum()}/{len(df)}")


Matches found: 19836/19988


In [10]:
df['has_5_UTR'] = df['5\' UTR'].notna() & (df['5\' UTR'] != '')
df['has_3_UTR'] = df['3\' UTR'].notna() & (df['3\' UTR'] != '')

df['5 UTR > 10 nt'] = df['5\' UTR_len'] > 10
df['CDS > 100 nt'] = df['CDS_len'] > 150

In [ ]:
# RiboNN header definitions:
# PREDICTED VALUES:
# - predicted_TE_[ID]:  The predicted Translational Efficiency (TE) for a specific 
#                       cell line or tissue. This represents the expected protein 
#                       output per mRNA molecule in that specific context.
# - mean_predicted_TE:  The average TE across all 70+ tested cell types/tissues; 
#                       serves as a baseline for general translational potential.
# BIOLOGICAL GROUPS IN THIS DATASET:
# - Cancer Lines:       A549 (Lung), HeLa (Cervical), HepG2 (Liver), MCF7 (Breast), 
#                       K562 (Leukemia), PC3 (Prostate),
# - Stem Cells:         H1.hESC, H9.hESC (Embryonic), WTC.11 (iPSC).
# - Primary/Normal:     Primary CD4+ T cells, primary macrophages, normal brain/
#                       kidney/prostate/muscle tissue.
# - Disease States:     HCC_tumor vs. HCC_adjacent_normal (Liver Cancer), 
#                       human_brain_tumor vs. normal_brain_tissue.
# - Differentiation:    Neuronal precursor cells, early_neurons, differentiated 
#                       dopamine neurons, megakaryocytes.

# utr5_sequence	cds_sequence	utr3_sequence

RiboNN_prediction_path = Path('../data/prediction_output.csv')
RiboNN_predictions = pd.read_csv(RiboNN_prediction_path, sep=',', header=0, comment='#', names=[
    'transcript_id', '5\' UTR', 'CDS', '3\' UTR', 'predicted_TE_108T', 'predicted_TE_12T', 'predicted_TE_A2780', 
    'predicted_TE_A549', 'predicted_TE_BJ', 'predicted_TE_BRx.142', 'predicted_TE_C643', 'predicted_TE_CRL.1634', 'predicted_TE_Calu.3', 
    'predicted_TE_Cybrid_Cells', 'predicted_TE_H1.hESC', 'predicted_TE_H1933', 'predicted_TE_H9.hESC', 'predicted_TE_HAP.1', 
    'predicted_TE_HCC_tumor', 'predicted_TE_HCC_adjancent_normal', 'predicted_TE_HCT116', 'predicted_TE_HEK293', 'predicted_TE_HEK293T', 
    'predicted_TE_HMECs', 'predicted_TE_HSB2', 'predicted_TE_HSPCs', 'predicted_TE_HeLa', 'predicted_TE_HeLa_S3', 'predicted_TE_HepG2', 
    'predicted_TE_Huh.7.5', 'predicted_TE_Huh7', 'predicted_TE_K562', ' predicted TE_Kidney_normal_tissue', ' predicted TE_LCL', 
    ' predicted TE_LuCaP.PDX', ' predicted TE_MCF10A', ' predicted TE_MCF10A.ER.Src', ' predicted TE_MCF7', ' predicted TE_MD55A3', 
    ' predicted TE_MDA.MB.231', ' predicted TE_MM1.S', ' predicted TE_MOLM.13',' predicted TE_Molt.3', ' predicted TE_Mutu', 
    ' predicted TE_OSCC', ' predicted TE_PANC1', ' predicted TE_PATU.8902', ' predicted TE_PC3', ' predicted TE_PC9', 
    ' predicted TE_Primary_CD4._T.cells', ' predicted TE_Primary_human_bronchial_epithelial_cells', ' predicted TE_RD.CCL.136', 
    ' predicted TE_RPE.1', ' predicted TE_SH.SY5Y', ' predicted TE_SUM159PT', ' predicted TE_SW480TetOnAPC', ' predicted TE_T47D', 
    ' predicted TE_THP.1', ' predicted TE_U.251', ' predicted TE_U.343', ' predicted TE_U2392', ' predicted TE_U2OS', ' predicted TE_Vero_6', 
    ' predicted TE_WI38', '_predicted_TE_WM902B', '_predicted_TE_WTC.11', '_predicted_TE_ZR75.1', '_predicted_TE_cardiac_fibroblasts', 
    '_predicted_TE_ccRCC', '_predicted_TE_early_neurons', '_predicted_TE_fibroblast', '_predicted_TE_hESC', '_predictedTE_human_brain_tumor', 
    '_predictedTE_iPSC.differentiated_dopamine_neurons', '_predictedTE_megakaryocytes', '_predictedTE_muscle_tissue', '_predictedTE_neuronal_precursor_cells',
    '_predictedTE_neurons', '_predictedTE_normal_brain_tissue', '_predictedTE_normal_prostate', '_predictedTE_primary_macrophages',
    '_predictedTE_skeletal_muscle','mean_predicted_TE'
])

cols_to_keep = [col for col in RiboNN_predictions.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, RiboNN_predictions[cols_to_keep], on='transcript_id', how='left')

In [16]:
df.columns

Index(['gene_id', 'transcript_id', '5' UTR', 'start_codon', 'CDS',
       'stop_codon', '3' UTR', 'appris_rank', '5' UTR_len', 'CDS_len',
       ...
       '_predictedTE_iPSC.differentiated_dopamine_neurons',
       '_predictedTE_megakaryocytes', '_predictedTE_muscle_tissue',
       '_predictedTE_neuronal_precursor_cells', '_predictedTE_neurons',
       '_predictedTE_normal_brain_tissue', '_predictedTE_normal_prostate',
       '_predictedTE_primary_macrophages', '_predictedTE_skeletal_muscle',
       'mean_predicted_TE'],
      dtype='str', length=117)